In [ ]:
!pip -q install kaggle

from google.colab import files
files.upload()

Saving kaggle.json to kaggle.json


{'kaggle.json': b'{"username": "amantyagi1", "key": "KGAT_4526b15d0cafb76563a6dc97ed0fb619"}'}

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d alessiocorrado99/animals10 -p /content
!unzip -q /content/animals10.zip -d /content/animals10
!ls /content/animals10

Dataset URL: https://www.kaggle.com/datasets/alessiocorrado99/animals10
License(s): GPL-2.0
 99% 579M/586M [00:03<00:00, 258MB/s]
100% 586M/586M [00:03<00:00, 175MB/s]
raw-img  translate.py


In [ ]:
import os, random, numpy as np, torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

Device: cuda


In [ ]:
data_dir = "/content/animals10/raw-img"
assert os.path.isdir(data_dir), f"Not found: {data_dir}. Check extracted folders."

IMG_SIZE = 224

train_tfms = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7,1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_tfms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

In [ ]:
full_for_split = datasets.ImageFolder(root=data_dir)
class_names = full_for_split.classes
num_classes = len(class_names)
print("Classes:", class_names)

targets = np.array(full_for_split.targets)
indices = np.arange(len(full_for_split))

train_idx, val_idx = train_test_split(
    indices,
    test_size=0.2,
    stratify=targets,
    random_state=SEED
)

train_ds = datasets.ImageFolder(root=data_dir, transform=train_tfms)
val_ds = datasets.ImageFolder(root=data_dir, transform=val_tfms)

train_ds = Subset(train_ds, train_idx)
val_ds = Subset(val_ds, val_idx)

print("Train size:", len(train_ds), "Val size:", len(val_ds))

Classes: ['cane', 'cavallo', 'elefante', 'farfalla', 'gallina', 'gatto', 'mucca', 'pecora', 'ragno', 'scoiattolo']
Train size: 20943 Val size: 5236


In [ ]:
train_targets = targets[train_idx]

class_counts = np.bincount(train_targets, minlength=num_classes)
print("Train class counts:", class_counts)

class_weights = 1.0 / np.clip(class_counts, a_min=1, a_max=None)
class_weights = class_weights / class_weights.sum() * num_classes

sample_weights = class_weights[train_targets]
sample_weights = torch.tensor(sample_weights, dtype=torch.double)

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=len(sample_weights),
    replacement=True
)

Train class counts: [3890 2098 1157 1690 2478 1334 1493 1456 3857 1490]


In [ ]:
ce_class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=ce_class_weights)
print("Class weights:", ce_class_weights)

Class weights: tensor([0.4566, 0.8466, 1.5352, 1.0510, 0.7168, 1.3315, 1.1897, 1.2199, 0.4605,
        1.1921], device='cuda:0')


In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE, sampler=sampler,
    num_workers=NUM_WORKERS, pin_memory=True
)

val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True
)

In [ ]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 47.8MB/s]


In [ ]:
def run_epoch(model, loader, train=True):
  model.train(train)
  total_loss, total_correct, total = 0.0, 0, 0

  for x, y in loader:
    x, y = x.to(device), y.to(device)

    with torch.set_grad_enabled(train):
      logits = model(x)
      loss = criterion(logits, y)

      if train:
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()

    total_loss += loss.item() * x.size(0)
    preds = logits.argmax(dim=1)
    total_correct += (preds == y).sum().item()
    total += x.size(0)

  return total_loss / total, total_correct / total

EPOCHS = 10
best_val_acc = 0.0

for epoch in range(1, EPOCHS +1):
  train_loss, train_acc = run_epoch(model, train_loader, train=True)
  val_loss, val_acc = run_epoch(model, val_loader, train=False)

  scheduler.step()

  print(f"Epoch {epoch:02d} | "
        f"train loss{train_loss:.4f} acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f}")

if val_acc > best_val_acc:
  best_val_acc = val_acc
  torch.save(model.state_dict(), "best_animals10_resnet18.pt")
  print("Model Saved")

Epoch 01 | train loss0.3194 acc 0.8889 | val loss 0.5636 acc 0.8109
Epoch 02 | train loss0.2176 acc 0.9198 | val loss 0.2563 acc 0.9139
Epoch 03 | train loss0.1633 acc 0.9415 | val loss 0.2537 acc 0.9181
Epoch 04 | train loss0.1170 acc 0.9553 | val loss 0.2454 acc 0.9219
Epoch 05 | train loss0.0990 acc 0.9618 | val loss 0.2090 acc 0.9312
Epoch 06 | train loss0.0686 acc 0.9731 | val loss 0.1746 acc 0.9482
Epoch 07 | train loss0.0437 acc 0.9830 | val loss 0.1554 acc 0.9565
Epoch 08 | train loss0.0282 acc 0.9880 | val loss 0.1586 acc 0.9595
Epoch 09 | train loss0.0223 acc 0.9904 | val loss 0.1455 acc 0.9605
Epoch 10 | train loss0.0184 acc 0.9922 | val loss 0.1438 acc 0.9633
Model Saved


In [ ]:
model.load_state_dict(torch.load("best_animals10_resnet18.pt", map_location=device))
model.eval()

all_preds, all_true = [], []

with torch.no_grad():
  for x, y in val_loader:
    x = x.to(device)
    logits = model(x)
    preds = logits.argmax(dim=1).cpu().numpy()
    all_preds.extend(preds)
    all_true.extend(y.numpy())

cm = confusion_matrix(all_true, all_preds)
print("Confusion Matrix:\n", cm)

print("\nClassification Report:")
print(classification_report(all_true, all_preds, target_names=class_names))

Confusion Matrix:
 [[922  17   1   2   5  11   3   5   2   5]
 [  3 509   0   1   1   0   5   4   0   2]
 [  4   2 274   2   0   0   3   2   0   2]
 [  1   1   0 416   0   0   0   0   4   0]
 [  7   2   0   0 602   2   1   0   3   3]
 [  6   0   0   0   0 324   1   0   1   2]
 [  3  13   4   0   5   0 340   8   0   0]
 [  2   3   1   0   1   1   7 346   0   3]
 [  1   2   1   8   1   1   0   0 947   3]
 [  2   0   0   0   2   2   0   1   1 364]]

Classification Report:
              precision    recall  f1-score   support

        cane       0.97      0.95      0.96       973
     cavallo       0.93      0.97      0.95       525
    elefante       0.98      0.95      0.96       289
    farfalla       0.97      0.99      0.98       422
     gallina       0.98      0.97      0.97       620
       gatto       0.95      0.97      0.96       334
       mucca       0.94      0.91      0.93       373
      pecora       0.95      0.95      0.95       364
       ragno       0.99      0.98      

In [ ]:
!pip -q install torchview graphviz
!apt-get install graphviz

from torchview import draw_graph

model_cpu = model.cpu()
graph = draw_graph(model_cpu, input_size=(1, 3, 224, 224), expand_nested=True)

graph.visual_graph.render("animals10_model_graph", format="png", cleanup=True)

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
graphviz is already the newest version (2.42.2-6ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.


'animals10_model_graph.png'